In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsdiegoadama2204";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://external@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/folder/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.clientes (
id_cliente string,
nombre_completo string,
fecha_nacimiento timestamp,
correo string,
ciudad string,
fecha_registro timestamp
--longitude double,
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/clientes_example"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.polizas (
id_poliza string,
id_cliente string,
tipo_seguro string,
fecha_inicio timestamp,
prima_anual double,
estado string
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/polizas_example"

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.bronze.siniestros (
  id_siniestro string,
  id_poliza string,
  fecha_siniestro timestamp,
  monto_reclamado double,
  estado_siniestro string
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/siniestros_example"

###Tablas Silver

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.silver.clientes_polizas_transformed (
  id_poliza string,
  id_cliente string,
  nombre_completo string,
  correo string,
  ciudad string,
  fecha_nacimiento timestamp,
  fecha_registro timestamp,
  fecha_inicio timestamp,
  tipo_seguro string,
  estado_poliza string,
  prima_anual double,
  edad_cliente integer,
  antiguedad_dias_cliente integer
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/clientes_polizas_transformed"

###Tablas Golden

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_kpis (
  mes timestamp,
  ciudad string,
  tipo_seguro string,
  polizas integer,
  siniestros integer,
  prima_anual_total double,
  monto_reclamado_total double,
  tasa_aprobacion_siniestro integer,
  ratio double
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_kpis"

